# Processing standard MOUSE files with MoDaCor

This notebook discovers MoDaCor-ready stacked MOUSE files, validates each sample/background pairing, previews the universal solids pipeline, and processes selected batches through reusable configuration-specific runtime sessions.

For new data, create the `*_stacked_modacor.nxs` files first with `mouse-stacked-to-modacor`. Run this notebook from top to bottom and edit only **Configuration** for normal use.


## Remaining uncertainty metadata

The pipeline propagates the uncertainties currently available in the converted files. Upstream estimates are still needed for total count time, incident flux, dark-current correction, flatfield values, detector coordinates/rotations, and sensor thickness. Detector readout time is an acquisition parameter and must not be used as total-count-time uncertainty.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("BAM/MOUSE")
sys.path.insert(0, str(PROJECT_DIR))

import atexit
import sys

from IPython.display import Markdown, display

import modacor
from modacor.client import LocalRuntimeServer
from modacor.runner.pipeline import Pipeline

from mouse_helpers import discover_measurement_pairs, source_registrations


## Configuration


In [ ]:
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "MOUSE_solids.yaml"
DATA_ROOT = PROJECT_DIR / "data"
SAMPLE_BATCH_START = 2
SAMPLE_BATCH_END = 2
OUTPUT_DIR = PROJECT_DIR / "work" / "output"

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"sample": ["signal"], "background": ["signal"]}}
OUTPUT_DATA_PATHS = ["/sample/signal", "/sample/Q"]


## Discover and validate measurement pairs


In [ ]:
measurement_pairs = discover_measurement_pairs(
    DATA_ROOT,
    batch_start=SAMPLE_BATCH_START,
    batch_end=SAMPLE_BATCH_END,
)
print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurement pairs: {len(measurement_pairs)}")
for pair in measurement_pairs:
    route = "displaced dispersant" if pair["use_dispersant_pipeline"] else "standard solids"
    print(f"  config {pair['configuration']}: {pair['sample'].name} ({route})")


## Preview the correction graph


In [ ]:
pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
display(Markdown(f"```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
print(f"Prepared {len(pipeline.graph)} steps from {PIPELINE_PATH.name}.")


## Start or reuse the runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Process the selected batches

Each instrument configuration receives one session. `mode="auto"` performs a full first run and reuses state for later sample/background pairs, without notebook-side state flags.


In [ ]:
sessions = {}
batch_results = []

for pair in measurement_pairs:
    if pair["use_dispersant_pipeline"]:
        raise NotImplementedError(f"{pair['sample'].name} requires the future displaced-dispersant pipeline")

    configuration = pair["configuration"]
    if configuration not in sessions:
        session = client.replace_session(
            f"mouse-{configuration}",
            name=f"MOUSE configuration {configuration}",
            source_profile="mouse",
            pipeline_yaml_path=str(PIPELINE_PATH),
            trace=TRACE,
        )
        session.register_sink({"ref": "plots", "type": "plotly_json", "location": "buffer://session"})
        sessions[configuration] = session
    session = sessions[configuration]
    session.register_sources(*source_registrations(pair))

    output = OUTPUT_DIR / f"{pair['sample'].stem}_result.h5"
    result = session.process(
        mode="auto",
        changed_sources=["sample", "background"],
        run_name=pair["sample"].stem,
        rollback_snapshot=False,
        write_hdf={"path": str(output), "data_paths": OUTPUT_DATA_PATHS},
    )
    batch_results.append({"pair": pair, "output": output, "result": result})
    print(f"config {configuration}: {result['status']} ({result['effective_mode']}) -> {output.name}")

print(f"Completed {len(batch_results)} pairs in {len(sessions)} sessions.")


## Live plots


In [ ]:
links = ["## Configuration plots"]
for configuration, session in sorted(sessions.items()):
    links.extend([
        f"### Configuration {configuration}",
        f"- [Corrected I(Q)]({session.plot_url('plots', 'mouse-1d')})",
        f"- [Corrected detector image]({session.plot_url('plots', 'mouse-2d')})",
    ])
display(Markdown("\n".join(links)))


## Cleanup

This stops only a process launched by the notebook. An independently started runtime remains available.


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
